# 09 · Cómo decidir «quién es» y calidad de las fotos de registro

Se ejecuta en **Google Colab** (GPU T4), de arriba abajo, con los mismos zips del notebook 06 (`ml.zip`, `luis.zip`, `aldana.zip`, `erick.zip`...).

**Dos preguntas, mismos embeddings:**

1. **¿Puntuar con la similitud máxima es lo mejor?** Hoy el servicio toma la foto registrada más parecida. Se compara con el promedio de las k más parecidas,
   el centroide de cada persona, un SVM y un k-NN. Como los puntajes de cada método tienen otra escala, se miden cosas que no dependen de ella:
   AUC entre propios y compañeros / extraños, y cuántos propios se reconocen si se deja pasar a lo sumo un 5 % de los compañeros parecidos.
2. **¿Las fotos borrosas explican los rechazos?** Se mide la nitidez de cada foto, su relación con el puntaje y si no registrar las menos nítidas mejora el reconocimiento.

No se reentrena nada (I6: no se suben fotos ni embeddings). Código: `ml/src/metodos_decision.py`.

In [ ]:
import os, zipfile
from google.colab import files

subidos = files.upload()     # ml.zip + un zip por persona
os.makedirs("/content/condohub-ia", exist_ok=True)
propios_dir = "/content/condohub-ml/datos/propios"
os.makedirs(propios_dir, exist_ok=True)
for nombre in subidos:
    z = zipfile.ZipFile(nombre)
    if nombre.lower().startswith("ml"):
        z.extractall("/content/condohub-ia")
        continue
    persona = os.path.splitext(nombre)[0].split(" (")[0].lower()
    con_raiz = all(n.startswith(persona + "/") for n in z.namelist())
    z.extractall(propios_dir if con_raiz else os.path.join(propios_dir, persona))
os.environ["ML_DATA_DIR"] = "/content/condohub-ml"
os.chdir("/content/condohub-ia")
print("personas encontradas:", sorted(os.listdir(propios_dir)))

In [ ]:
!pip install -q facenet-pytorch --no-deps
!pip install -q scikit-learn pyyaml matplotlib pillow
import torch
print("GPU disponible:", torch.cuda.is_available())

## 1 · Fotos, embeddings y nitidez

In [ ]:
import json, yaml, numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from ml.src import embeddings, lfw, metodos_decision as md, propios as fotos_propias, zonas
from ml.src.paths import dir_datos, dir_experimento

cfg = yaml.safe_load(open("ml/configs/registro_equipo.yaml", encoding="utf-8"))
raiz = dir_datos() / cfg["carpeta_propios"]
fotos = fotos_propias.listar_fotos(raiz)      # lanza SinConsentimiento si falta algún consentimiento.json (I1)

embeber = embeddings.crear_embebedor()
imagenes = [fotos_propias.cargar_imagen(f.ruta) for f in fotos]
nitidez_todas = np.array([md.nitidez(np.asarray(im)) for im in imagenes])     # antes de liberar las imágenes
emb, detectado = embeber(imagenes)
del imagenes

personas = np.array([f.persona for f in fotos])[detectado]
sesiones = np.array([f.sesion for f in fotos])[detectado]
nitidez = nitidez_todas[detectado]
emb = emb[detectado]
print("fotos usables:", len(emb), "|", dict(Counter(personas)))

In [ ]:
img_lfw, et_lfw, _ = lfw.cargar_lfw(cfg["lfw_min_fotos"])
rng = np.random.default_rng(cfg["semilla"])
elegidas = np.concatenate([rng.permutation(np.flatnonzero(et_lfw == p))[:cfg["lfw_fotos_por_persona"]]
                           for p in np.unique(et_lfw)])
emb_lfw, det_lfw = embeber([img_lfw[i] for i in elegidas])
emb_lfw, et_lfw = emb_lfw[det_lfw], et_lfw[elegidas][det_lfw]
del img_lfw
print(f"{len(emb_lfw)} fotos de {len(np.unique(et_lfw))} extraños")

## 2 · Pregunta 1: ¿qué método decide mejor?

Mismo protocolo que el notebook 06 (una sesión fuera, compañeros no registrados, extraños de LFW de prueba). Se espera que el **SVM y el k-NN** discriminen peor a los
extraños: un clasificador de conjunto cerrado siempre elige a alguien, y I2 necesita saber cuándo *no* es nadie.

In [ ]:
metodos = {
    "máximo (actual)": md.maximo,
    "top-3 promedio": md.top_k_promedio(3),
    "centroide": md.centroide,
    "SVM lineal": md.svm(),
    "k-NN (k=5)": md.knn(5),
}
tabla, puntos_por_metodo = {}, {}
for nombre, metodo in metodos.items():
    p = md.puntajes_equipo_con(metodo, emb, personas, sesiones, emb_lfw, et_lfw, cfg["semilla"])
    puntos_por_metodo[nombre] = p
    tabla[nombre] = md.resumen(p)

col = "reconocidos_con_5pct_companeros"
print(f"{'método':<18}{'identidad ok':>13}{'AUC prop/comp':>15}{'AUC prop/extr':>15}{'reconoc. @5% comp':>19}{'umbral':>9}")
for nombre, r in tabla.items():
    print(f"{nombre:<18}{r['identidad_correcta']:>13.3f}{r['auc_propios_vs_companeros']:>15.3f}{r['auc_propios_vs_extranos']:>15.3f}"
          f"{r[col]:>19.3f}{r['umbral_en_ese_punto']:>9.3f}")

## 3 · Pregunta 2: ¿explica la nitidez los rechazos?

Se usa el método actual (máximo). Si las fotos borrosas puntúan más bajo, la correlación (Spearman) entre nitidez y puntaje será positiva.
Luego se mide qué pasa si **no se registran** las fotos menos nítidas de cada persona (siguen usándose para probar).

In [ ]:
p = puntos_por_metodo["máximo (actual)"]
nit_prueba = nitidez[p["genuinos_indice"]]
rho = md.correlacion_rangos(nit_prueba, p["genuinos"])
print(f"Spearman nitidez vs puntaje de los propios: {rho:.3f}  ({len(nit_prueba)} fotos)")
bajos = p["genuinos"] < 0.70
print(f"nitidez mediana de las fotos con puntaje < 0,70: {np.median(nit_prueba[bajos]):.1f}  |  del resto: {np.median(nit_prueba[~bajos]):.1f}")

filas = []
for descarte in (0, 10, 20, 30):
    usable = np.ones(len(emb), dtype=bool)
    for persona in np.unique(personas):
        de_ella = personas == persona
        if descarte:
            usable[de_ella & (nitidez < np.percentile(nitidez[de_ella], descarte))] = False
    q = md.puntajes_equipo_con(md.maximo, emb, personas, sesiones, emb_lfw, et_lfw, cfg["semilla"], usable_en_registro=usable)
    r = md.resumen(q)
    filas.append({"descarte_pct": descarte, "fotos_registradas": int(usable.sum()), **r,
                  "propios_con_puntaje_bajo_070": float((q["genuinos"] < 0.70).mean())})
    print(f"descartar el {descarte:>2} % menos nítido del registro: {int(usable.sum()):>3} fotos | identidad ok {r['identidad_correcta']:.3f} | "
          f"@5% comp {r[col]:.3f} | AUC prop/comp {r['auc_propios_vs_companeros']:.3f} | propios < 0,70: {filas[-1]['propios_con_puntaje_bajo_070']:.3f}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(nit_prueba, p["genuinos"], s=10, alpha=0.6)
ejes[0].axhline(0.70, color="k", ls="--", lw=0.8); ejes[0].set_xscale("log")
ejes[0].set_xlabel("nitidez (varianza del laplaciano, log)"); ejes[0].set_ylabel("similitud con el perfil"); ejes[0].set_title(f"Nitidez vs puntaje (Spearman {rho:.2f})")
nombres = list(tabla)
ejes[1].bar(range(len(nombres)), [tabla[n][col] for n in nombres], color="tab:green")
ejes[1].set_xticks(range(len(nombres))); ejes[1].set_xticklabels(nombres, rotation=25, ha="right", fontsize=8)
ejes[1].set_ylabel("propios reconocidos con ≤5 % de compañeros aceptados"); ejes[1].set_ylim(0, 1)
fig.tight_layout()
destino = dir_experimento("decision_y_calidad")
fig.savefig(destino / "decision_y_calidad.png", dpi=150); plt.show()

## 4 · Guardar y descargar

Solo métricas y una figura (sin fotos ni embeddings, I1/I6).

In [ ]:
from google.colab import files

resultado = {"config": cfg, "fotos_usables": int(len(emb)), "personas": {k: int(v) for k, v in Counter(personas).items()},
             "extranos_lfw": {"fotos": int(len(emb_lfw)), "personas": int(len(np.unique(et_lfw)))},
             "metodos": tabla, "nitidez": {"spearman_con_puntaje": rho, "descarte_en_el_registro": filas}}
(destino / "metrics.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")
for nombre in ("metrics.json", "decision_y_calidad.png"):
    files.download(str(destino / nombre))